# Customer Portfolio Analysis and Segmentation

This project analyzes customer behavior using transactional sales data from a wholesale distribution business.

The goal is to move beyond a simple historical customer list and identify the **active and commercially relevant customer portfolio**, measure revenue concentration, segment customers, and detect recovery opportunities.

## 1. Business Problem

Wholesale companies often accumulate large customer databases over time. However, not every historical customer remains commercially relevant.

This analysis aims to answer:

- How many customers are actually active today?
- How much revenue is concentrated in the active customer portfolio?
- Which customers belong to the most valuable segment?
- How concentrated is customer revenue?
- Are there high-value customers that became inactive?

The analysis uses customer segmentation techniques such as **Pareto / ABC segmentation**, **activity classification**, and **RFM-style scoring**.

## 2. Dataset

The dataset contains transactional sales records with the following main fields:

- Sale date
- Salesperson
- Customer
- Document type
- Invoice / document number
- Product
- Quantity
- Unit price
- Net amount
- VAT
- Total amount

For this customer analysis, only invoiced customers are considered. Generic retail sales assigned to `Público general` are excluded because they do not represent identifiable customer accounts.

> **Data privacy note:** The original dataset contains real business transactions. For a public GitHub repository, use an anonymized or sample version of the data.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 3. Configuration

The notebook uses relative paths so it can be used inside a GitHub repository.

Expected structure:

```text
customer-portfolio-analysis/
├── data/
│   └── detalle_venta.xlsx
├── images/
├── notebooks/
│   └── customer_analytics.ipynb
└── requirements.txt
```

In [ ]:
# Update this path if needed
DATA_PATH = Path("../data/detalle_venta.xlsx")
IMAGES_DIR = Path("../images")
OUTPUTS_DIR = Path("../outputs")

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

DAYS_ACTIVE_CUSTOMER = 120
MIN_DOCUMENTS_RECURRENT = 3

## 4. Data Preparation

The preparation step standardizes column names, converts dates and numeric fields, removes invalid transactions, and creates time-based variables for later analysis.

In [ ]:
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("á", "a", regex=False)
        .str.replace("é", "e", regex=False)
        .str.replace("í", "i", regex=False)
        .str.replace("ó", "o", regex=False)
        .str.replace("ú", "u", regex=False)
        .str.replace("ñ", "n", regex=False)
    )
    return df


def load_sales_data(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path)
    df = clean_column_names(df)

    df["Fecha_Venta"] = pd.to_datetime(df["Fecha_Venta"], errors="coerce")

    numeric_columns = ["Cantidad", "Precio_Unitario", "Neto", "IVA", "Total"]
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    text_columns = ["Vendedor", "Cliente", "Tipo", "Producto"]
    for col in text_columns:
        df[col] = df[col].astype(str).str.strip()

    df["Cliente"] = df["Cliente"].replace(["nan", "None", ""], "Público general")

    df = df.dropna(subset=["Fecha_Venta", "Producto", "Total"])
    df = df[df["Total"] > 0].copy()

    df["Año"] = df["Fecha_Venta"].dt.year
    df["Mes"] = df["Fecha_Venta"].dt.month
    df["Periodo"] = df["Fecha_Venta"].dt.to_period("M").astype(str)

    return df

In [ ]:
df_sales = load_sales_data(DATA_PATH)

print(f"Rows: {df_sales.shape[0]:,}")
print(f"Columns: {df_sales.shape[1]:,}")
df_sales.head()

## 5. Customer Dataset

The analysis excludes generic public sales and keeps only invoice-based transactions. This provides a cleaner customer-level dataset.

In [ ]:
df_customers_tx = df_sales[
    (df_sales["Cliente"] != "Público general")
    &
    (df_sales["Tipo"].str.contains("FACTURA", case=False, na=False))
].copy()

print(f"Customer transaction rows: {df_customers_tx.shape[0]:,}")
print(f"Identified customers: {df_customers_tx['Cliente'].nunique():,}")

## 6. Customer Aggregation

Each customer is summarized using revenue, number of documents, product variety, first purchase, last purchase, and recency.

In [ ]:
def build_customer_summary(df_customers_tx: pd.DataFrame) -> pd.DataFrame:
    max_date = df_customers_tx["Fecha_Venta"].max()

    customers = (
        df_customers_tx
        .groupby("Cliente")
        .agg(
            venta_total=("Total", "sum"),
            venta_neta=("Neto", "sum"),
            iva_total=("IVA", "sum"),
            cantidad_total=("Cantidad", "sum"),
            documentos=("Folio", "nunique"),
            productos_distintos=("Producto", "nunique"),
            primera_compra=("Fecha_Venta", "min"),
            ultima_compra=("Fecha_Venta", "max")
        )
        .reset_index()
    )

    customers["ticket_promedio"] = customers["venta_total"] / customers["documentos"]
    customers["dias_desde_ultima_compra"] = (max_date - customers["ultima_compra"]).dt.days

    return customers


def recalculate_abc(customers: pd.DataFrame) -> pd.DataFrame:
    customers = customers.copy().sort_values("venta_total", ascending=False)

    customers["participacion_%"] = customers["venta_total"] / customers["venta_total"].sum() * 100
    customers["participacion_acumulada_%"] = customers["participacion_%"].cumsum()

    customers["segmento_abc"] = np.where(
        customers["participacion_acumulada_%"] <= 80,
        "A",
        np.where(customers["participacion_acumulada_%"] <= 95, "B", "C")
    )

    return customers

In [ ]:
customers_historical = build_customer_summary(df_customers_tx)
customers_historical = recalculate_abc(customers_historical)

customers_historical.head()

## 7. Customer Activity and Relevance Criteria

A customer is classified as **active** when their last purchase occurred within the last 120 days of the dataset.

A customer is classified as **active relevant** when they are active and also meet at least one of the following conditions:

- Recurrent customer: 3 or more invoices
- High-value customer: ABC Segment A in the historical portfolio

This avoids treating one-time or low-value customers as part of the current commercial portfolio, while still preserving high-value institutional customers that may purchase less frequently.

In [ ]:
customers_historical["estado_actividad"] = np.where(
    customers_historical["dias_desde_ultima_compra"] <= DAYS_ACTIVE_CUSTOMER,
    "Activo",
    "Inactivo"
)

customers_historical["tipo_frecuencia"] = np.where(
    customers_historical["documentos"] >= MIN_DOCUMENTS_RECURRENT,
    "Recurrente",
    "Ocasional"
)

customers_historical["es_alto_valor"] = np.where(
    customers_historical["segmento_abc"] == "A",
    "Sí",
    "No"
)

customers_historical["perfil_cliente"] = np.select(
    [
        (customers_historical["estado_actividad"] == "Activo") &
        (customers_historical["tipo_frecuencia"] == "Recurrente") &
        (customers_historical["es_alto_valor"] == "Sí"),

        (customers_historical["estado_actividad"] == "Activo") &
        (customers_historical["tipo_frecuencia"] == "Ocasional") &
        (customers_historical["es_alto_valor"] == "Sí"),

        (customers_historical["estado_actividad"] == "Activo") &
        (customers_historical["tipo_frecuencia"] == "Recurrente") &
        (customers_historical["es_alto_valor"] == "No"),

        (customers_historical["estado_actividad"] == "Activo") &
        (customers_historical["tipo_frecuencia"] == "Ocasional") &
        (customers_historical["es_alto_valor"] == "No"),

        (customers_historical["estado_actividad"] == "Inactivo") &
        (customers_historical["es_alto_valor"] == "Sí"),
    ],
    [
        "Activo recurrente alto valor",
        "Activo ocasional alto valor",
        "Activo recurrente",
        "Activo ocasional",
        "Alto valor inactivo",
    ],
    default="Inactivo bajo/medio valor"
)

customers_active = customers_historical[
    customers_historical["estado_actividad"] == "Activo"
].copy()

customers_active = recalculate_abc(customers_active)

customers_active_relevant = customers_historical[
    (customers_historical["estado_actividad"] == "Activo")
    &
    (
        (customers_historical["tipo_frecuencia"] == "Recurrente")
        |
        (customers_historical["es_alto_valor"] == "Sí")
    )
].copy()

customers_active_relevant = recalculate_abc(customers_active_relevant)

print("Historical customers:", customers_historical["Cliente"].nunique())
print("Active customers:", customers_active["Cliente"].nunique())
print("Active relevant customers:", customers_active_relevant["Cliente"].nunique())

## 8. RFM-Style Scoring

RFM scoring is used as an additional customer segmentation layer:

- **Recency:** how recently the customer purchased
- **Frequency:** how often the customer purchased
- **Monetary:** how much revenue the customer generated

In [ ]:
def add_rfm_segmentation(customers: pd.DataFrame) -> pd.DataFrame:
    customers = customers.copy()

    customers["score_recencia"] = pd.qcut(
        customers["dias_desde_ultima_compra"].rank(method="first"),
        5,
        labels=[5, 4, 3, 2, 1]
    ).astype(int)

    customers["score_frecuencia"] = pd.qcut(
        customers["documentos"].rank(method="first"),
        5,
        labels=[1, 2, 3, 4, 5]
    ).astype(int)

    customers["score_monetario"] = pd.qcut(
        customers["venta_total"].rank(method="first"),
        5,
        labels=[1, 2, 3, 4, 5]
    ).astype(int)

    customers["rfm_score"] = (
        customers["score_recencia"].astype(str)
        + customers["score_frecuencia"].astype(str)
        + customers["score_monetario"].astype(str)
    )

    customers["rfm_total"] = (
        customers["score_recencia"]
        + customers["score_frecuencia"]
        + customers["score_monetario"]
    )

    def classify_customer(row):
        if row["score_recencia"] >= 4 and row["score_frecuencia"] >= 4 and row["score_monetario"] >= 4:
            return "Cliente estrella"
        elif row["score_recencia"] <= 2 and row["score_monetario"] >= 4:
            return "Cliente valioso en riesgo"
        elif row["score_recencia"] >= 4 and row["score_frecuencia"] <= 2:
            return "Cliente nuevo o poco frecuente"
        elif row["score_recencia"] <= 2 and row["score_frecuencia"] <= 2:
            return "Cliente inactivo o perdido"
        elif row["score_frecuencia"] >= 4 and row["score_monetario"] >= 3:
            return "Cliente recurrente"
        else:
            return "Cliente regular"

    customers["segmento_rfm"] = customers.apply(classify_customer, axis=1)

    return customers

customers_historical = add_rfm_segmentation(customers_historical)
customers_active = add_rfm_segmentation(customers_active)
customers_active_relevant = add_rfm_segmentation(customers_active_relevant)

## 9. Portfolio Overview

The first comparison shows how many customers remain commercially relevant and how much revenue they represent.

In [ ]:
def portfolio_summary(name: str, customers: pd.DataFrame, historical_revenue: float) -> dict:
    return {
        "portfolio": name,
        "customers": customers["Cliente"].nunique(),
        "revenue": customers["venta_total"].sum(),
        "net_revenue": customers["venta_neta"].sum(),
        "documents": customers["documentos"].sum(),
        "avg_ticket": customers["venta_total"].sum() / customers["documentos"].sum(),
        "share_of_historical_revenue": customers["venta_total"].sum() / historical_revenue * 100
    }

historical_revenue = customers_historical["venta_total"].sum()

portfolio_overview = pd.DataFrame([
    portfolio_summary("Historical portfolio", customers_historical, historical_revenue),
    portfolio_summary("Active portfolio", customers_active, historical_revenue),
    portfolio_summary("Active relevant portfolio", customers_active_relevant, historical_revenue),
])

portfolio_overview

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(portfolio_overview["portfolio"], portfolio_overview["customers"])
plt.title("Customer Count by Portfolio View")
plt.ylabel("Customers")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "customer_count_by_portfolio.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(portfolio_overview["portfolio"], portfolio_overview["share_of_historical_revenue"])
plt.title("Revenue Share by Portfolio View")
plt.ylabel("Share of historical revenue (%)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "revenue_share_by_portfolio.png", dpi=150)
plt.show()

### Key Finding

The active relevant portfolio represents a smaller share of the historical customer base, but it concentrates most of the revenue. This validates the need to distinguish between historical customers and the current commercial portfolio.

## 10. ABC Segmentation of Active Relevant Customers

ABC segmentation is recalculated on the active relevant portfolio to better reflect the current business structure.

In [ ]:
abc_summary = (
    customers_active_relevant
    .groupby("segmento_abc")
    .agg(
        customers=("Cliente", "count"),
        revenue=("venta_total", "sum"),
        avg_revenue=("venta_total", "mean"),
        median_revenue=("venta_total", "median"),
        avg_ticket=("ticket_promedio", "mean"),
        avg_documents=("documentos", "mean"),
        avg_products=("productos_distintos", "mean")
    )
    .reset_index()
)

abc_summary["customer_share_%"] = abc_summary["customers"] / abc_summary["customers"].sum() * 100
abc_summary["revenue_share_%"] = abc_summary["revenue"] / abc_summary["revenue"].sum() * 100
abc_summary = abc_summary.sort_values("segmento_abc")

abc_summary

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(abc_summary["segmento_abc"], abc_summary["revenue_share_%"])
plt.title("Revenue Share by ABC Segment")
plt.xlabel("ABC Segment")
plt.ylabel("Revenue share (%)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "abc_revenue_share.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(abc_summary["segmento_abc"], abc_summary["customer_share_%"])
plt.title("Customer Share by ABC Segment")
plt.xlabel("ABC Segment")
plt.ylabel("Customer share (%)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "abc_customer_share.png", dpi=150)
plt.show()

### Key Finding

A small group of Segment A customers generates most of the revenue in the active relevant portfolio. This confirms a strong Pareto pattern in the customer base.

## 11. Segment Comparison

The next table compares customer segments by revenue, ticket size, frequency, and product variety.

In [ ]:
segment_comparison = abc_summary.copy()

c_avg_revenue = segment_comparison.loc[segment_comparison["segmento_abc"] == "C", "avg_revenue"].values[0]
c_avg_ticket = segment_comparison.loc[segment_comparison["segmento_abc"] == "C", "avg_ticket"].values[0]
c_avg_documents = segment_comparison.loc[segment_comparison["segmento_abc"] == "C", "avg_documents"].values[0]

segment_comparison["avg_revenue_vs_C"] = segment_comparison["avg_revenue"] / c_avg_revenue
segment_comparison["avg_ticket_vs_C"] = segment_comparison["avg_ticket"] / c_avg_ticket
segment_comparison["avg_documents_vs_C"] = segment_comparison["avg_documents"] / c_avg_documents

segment_comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(segment_comparison["segmento_abc"], segment_comparison["avg_revenue"])
plt.title("Average Revenue by ABC Segment")
plt.xlabel("ABC Segment")
plt.ylabel("Average revenue")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "avg_revenue_by_segment.png", dpi=150)
plt.show()

## 12. Pareto Curve

The Pareto curve shows the cumulative contribution of customers to total revenue.

In [ ]:
pareto_df = customers_active_relevant.sort_values("venta_total", ascending=False).copy()
pareto_df["customer_rank"] = range(1, len(pareto_df) + 1)
pareto_df["customer_share_cumulative_%"] = pareto_df["customer_rank"] / len(pareto_df) * 100
pareto_df["revenue_share_cumulative_%"] = pareto_df["venta_total"].cumsum() / pareto_df["venta_total"].sum() * 100

plt.figure(figsize=(8, 5))
plt.plot(pareto_df["customer_share_cumulative_%"], pareto_df["revenue_share_cumulative_%"])
plt.axhline(80, linestyle="--", linewidth=1)
plt.title("Customer Pareto Curve - Active Relevant Portfolio")
plt.xlabel("Cumulative customer share (%)")
plt.ylabel("Cumulative revenue share (%)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "customer_pareto_curve.png", dpi=150)
plt.show()

## 13. Revenue Distribution

Customer revenue is usually highly skewed in wholesale businesses. Percentiles help quantify that concentration.

In [ ]:
revenue_percentiles = pd.DataFrame({
    "percentile": ["P25", "P50", "P75", "P90", "P95", "P99"],
    "revenue": [
        customers_active_relevant["venta_total"].quantile(0.25),
        customers_active_relevant["venta_total"].quantile(0.50),
        customers_active_relevant["venta_total"].quantile(0.75),
        customers_active_relevant["venta_total"].quantile(0.90),
        customers_active_relevant["venta_total"].quantile(0.95),
        customers_active_relevant["venta_total"].quantile(0.99),
    ]
})

revenue_percentiles

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(revenue_percentiles["percentile"], revenue_percentiles["revenue"])
plt.title("Revenue Percentiles - Active Relevant Customers")
plt.xlabel("Percentile")
plt.ylabel("Revenue")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "revenue_percentiles.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(customers_active_relevant["venta_total"], bins=40)
plt.title("Revenue Distribution - Active Relevant Customers")
plt.xlabel("Customer revenue")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "revenue_distribution.png", dpi=150)
plt.show()

## 14. Inactive High-Value Customers

High-value inactive customers represent a direct recovery opportunity. These are customers that historically belonged to the top revenue segment but have not purchased recently.

In [ ]:
high_value_inactive = customers_historical[
    customers_historical["perfil_cliente"] == "Alto valor inactivo"
].sort_values("venta_total", ascending=False)

high_value_inactive_summary = pd.DataFrame({
    "metric": [
        "High-value inactive customers",
        "Historical revenue",
        "Average historical revenue",
        "Median historical revenue",
        "Average days since last purchase",
        "Median days since last purchase",
    ],
    "value": [
        len(high_value_inactive),
        high_value_inactive["venta_total"].sum(),
        high_value_inactive["venta_total"].mean(),
        high_value_inactive["venta_total"].median(),
        high_value_inactive["dias_desde_ultima_compra"].mean(),
        high_value_inactive["dias_desde_ultima_compra"].median(),
    ]
})

high_value_inactive_summary

In [ ]:
plt.figure(figsize=(9, 5))
plot_data = high_value_inactive.head(10).copy()
plot_data["Cliente_Anon"] = [f"Customer {i+1}" for i in range(len(plot_data))]
plt.barh(plot_data["Cliente_Anon"], plot_data["venta_total"])
plt.title("Top Inactive High-Value Customers")
plt.xlabel("Historical revenue")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "inactive_high_value_customers.png", dpi=150)
plt.show()

## 15. Product Preference by Customer

This table identifies the top product for each customer based on revenue contribution. It can support cross-selling or customer-specific commercial strategies.

In [ ]:
customer_product = (
    df_customers_tx
    .groupby(["Cliente", "Producto"])
    .agg(
        revenue=("Total", "sum"),
        quantity=("Cantidad", "sum"),
        documents=("Folio", "nunique")
    )
    .reset_index()
    .sort_values(["Cliente", "revenue"], ascending=[True, False])
)

favorite_product_by_customer = (
    customer_product
    .groupby("Cliente")
    .head(1)
    .rename(columns={"Producto": "favorite_product", "revenue": "favorite_product_revenue"})
)

favorite_product_by_customer.head()

## 16. Export Analytical Outputs

The final Excel file contains the main tables used for reporting, dashboarding, or Power BI modeling.

In [ ]:
output_file = OUTPUTS_DIR / "customer_analytics_outputs.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    portfolio_overview.to_excel(writer, sheet_name="portfolio_overview", index=False)
    customers_historical.to_excel(writer, sheet_name="customers_historical", index=False)
    customers_active.to_excel(writer, sheet_name="customers_active", index=False)
    customers_active_relevant.to_excel(writer, sheet_name="active_relevant", index=False)
    abc_summary.to_excel(writer, sheet_name="abc_summary", index=False)
    segment_comparison.to_excel(writer, sheet_name="segment_comparison", index=False)
    revenue_percentiles.to_excel(writer, sheet_name="revenue_percentiles", index=False)
    high_value_inactive.to_excel(writer, sheet_name="high_value_inactive", index=False)
    high_value_inactive_summary.to_excel(writer, sheet_name="inactive_summary", index=False)
    favorite_product_by_customer.to_excel(writer, sheet_name="favorite_product", index=False)

print(f"Output exported to: {output_file}")

## 17. Conclusions

Main findings from the analysis:

- A large share of historical customers are not part of the current active commercial portfolio.
- The active relevant customer portfolio concentrates most of the revenue.
- Revenue follows a strong Pareto pattern: a small group of customers generates most sales.
- Segment A customers outperform Segment C customers in revenue, purchase frequency, and product variety.
- Inactive high-value customers represent a concrete commercial recovery opportunity.

## Recommended Business Actions

- Prioritize retention actions for Segment A customers.
- Develop growth campaigns for Segment B customers.
- Automate low-touch communication for Segment C customers.
- Create a recovery campaign for inactive high-value customers.
- Use product preference data to support cross-selling and account management.